In [6]:
import os
import pandas as pd

In [71]:
'''
function: SCISSORS(dirpath)

arguments: 
    dirpath - path to directory of CBC.csv files generated by mahimahi script. 

description: 
    function to process CBC.csv for strand-specific counting. 
'''

def SCISSORS(dirpath):
    C=0
    filelist=[i for i in os.listdir(dirpath) if i.endswith("CBC.csv") ]    
    print("Reading:"+ str(filelist))
    for i in filelist:
        print(i)
        iSAM=pd.read_table(dirpath+i,sep=",")
        
        # Filter for good reads with clear mapping to for or rev strand.
        iSAM=iSAM[iSAM["flag"].isin([0,16])]

        # # # #
        # Annotate reads
        # # # #
        iSAM["det_strand"]=["Neg" if i==0 else "Pos" for i in iSAM["flag"]] #strand.

        iSAM["sample"]=i #sample
        iSAM["datalabel"]=i.split("_S")[0] # creat a common label for genotypes (across sequenced samples) as 'datalabel'
        
        # # # #
        # Count reads
        # # # #
        
        # Count reads based on Cell barcode, UMI, and strand. 
        iSAM=iSAM.join(iSAM.groupby("CBC")["CBC"].size(), on="CBC", rsuffix="_readcount")

        # Number of unique UMIs per cell
        iSAM=iSAM.join(iSAM.groupby(["CBC"])["UMI"].nunique(), on=["CBC"], rsuffix="_count")
        
        # Number of unique UMIs per strand per cell
        iSAM=iSAM.join(iSAM.groupby(["CBC","flag"])["UMI"].nunique(), on=["CBC","flag"], rsuffix="_strand_count")

        # # # #
        # Reshape data
        # # # # 
        AggCounts=iSAM.join(iSAM.pivot_table(index="CBC",columns=["det_strand"],values="UMI_strand_count",fill_value=0),on=["CBC"])
        AggCounts=AggCounts.assign(Neg_PosRatio=lambda x: x.Neg/((x.Pos)+1))
        AggCounts=AggCounts.assign(Rep_Index=lambda x: x.Neg/(x.Pos+x.Neg))
        AggCounts=pd.concat([AggCounts[AggCounts["CBC_readcount"]>100].drop("UMI",axis=1).iloc[:,12:],AggCounts[AggCounts["CBC_readcount"]>100].drop("UMI",axis=1).iloc[:,3]], axis=1)
        AggCounts=AggCounts.drop_duplicates()
        #print(AggCounts)
        #print(pd.melt(AggCounts,var_name="strand",value_name="strand_readcount",id_vars=["datalabel","CBC","ref_name","CBC_readcount","Rep_Index","Neg_PosRatio"], value_vars=["Neg","Pos"]))
        #print(AggCounts)
        # Output to csv
        AggCounts.to_csv(i.replace(".csv","_SCISSORS.csv")) 
        #break
        #print("Wrote "+i.replace(".csv","_SCISSORS.csv"))

    print('done.')


In [72]:
EV71=SCISSORS("/hpcdata/lvd_qve/Projects/PTD_StrandSpecificCounting_scRNAseq/81123_SCISSORS_FreshAnalysis/EV71/")

Reading:['EVA71_TT_S4_CBC.csv', 'EVA71_HH_S3_CBC.csv', 'Mock_5h_S5_CBC.csv']
EVA71_TT_S4_CBC.csv
EVA71_HH_S3_CBC.csv
Mock_5h_S5_CBC.csv
done.


In [73]:
CVB=SCISSORS("/hpcdata/lvd_qve/Projects/PTD_StrandSpecificCounting_scRNAseq/81123_SCISSORS_FreshAnalysis/CVB/")

Reading:['CVB3_HH_S1_CBC.csv', 'Mock_5h_S5_CBC.csv', 'CVB3_TT_S2_CBC.csv']
CVB3_HH_S1_CBC.csv
Mock_5h_S5_CBC.csv
CVB3_TT_S2_CBC.csv
done.


In [66]:
PV=SCISSORS("/hpcdata/lvd_qve/Projects/PTD_StrandSpecificCounting_scRNAseq/81123_SCISSORS_FreshAnalysis/PV_Rep/")

Reading:['RFP_488P_S3_CBC.csv', 'WT_IRES_mRuby3_MutPol_WT_IRES_GFP_S6_CBC.csv', 'WT_IRES_GFP_S1_CBC.csv', 'Del_IRES_mRuby3_MutPol_S5_CBC.csv', 'WT_IRES_mRuby3_MutPol_S4_CBC.csv', 'RFP_C109S_S2_CBC.csv', 'WT_GFP_S1_CBC.csv', 'WT_GFP_RFP_488P_S6_CBC.csv', 'WT_GFP_RFP_D177A_S7_CBC.csv', 'RFPMutPol_S3_CBC.csv', 'RFP_D177A_S4_CBC.csv', 'WT_GFP_RFP_C109S_S5_CBC.csv', 'WT_IRES_GFP_Del_IRES_mRuby3_S3_CBC.csv', 'WT_GFP_RFPMutPol_S2_CBC.csv', 'Del_IRES_mRuby3_S2_CBC.csv']
RFP_488P_S3_CBC.csv
                      CBC det_strand               sample datalabel  \
5        TGGGAGTCAGTAAGCG        Pos  RFP_488P_S3_CBC.csv  RFP_488P   
10       AGTTGGTGTATGGTTC        Pos  RFP_488P_S3_CBC.csv  RFP_488P   
11       CTTCTCTCAGTCCTTC        Pos  RFP_488P_S3_CBC.csv  RFP_488P   
15       TTGGAACGTCCCTACT        Pos  RFP_488P_S3_CBC.csv  RFP_488P   
16       CGGAGTCTCATGTCCC        Pos  RFP_488P_S3_CBC.csv  RFP_488P   
...                   ...        ...                  ...       ...   
9357001  ACACCCT